In [66]:
import pandas as pd
import numpy as np
import os 

In [67]:
#razlikovace se jer je za xgb label encoded, a za dnn one hot
file = 'features_for_XGBoost.csv'
if os.path.exists(file):
    sve = pd.read_csv(file)
else:
    sve = pd.DataFrame()

In [68]:
trke = pd.read_csv('all_races.csv')

In [69]:
trke.columns


Index(['season', 'round', 'race_name', 'date', 'time', 'circuit', 'country'], dtype='object')

In [70]:
trke = trke.drop(['date', 'time', 'country'], axis=1)

In [71]:
staze = set(zip(trke['circuit']))

In [72]:
staze = pd.DataFrame(staze)

In [73]:
staze.to_csv('non_processed_circuits.csv', index=False)
print(f"Sacuvano")

Sacuvano


In [74]:
staze = pd.read_csv('processed_circuits.csv')

In [76]:
trke = trke.merge(staze, on='circuit', how='left')    
    

In [79]:
vreme = pd.read_csv('all_weather.csv')
vreme = vreme.drop(['date', 'time', 'race_name'], axis=1)

In [82]:
#primenjujemo binning za kisu
bins = [ -1, 0, 1, 5, 100 ]
labels = [0, 1, 2, 3]
vreme['rain_bin'] = pd.cut(vreme['Rainfall'], bins=bins, labels=labels)


In [87]:
#binning za jedan sample vetra
def wind_dir_bin(deg):
    if (deg >= 337.5) or (deg < 22.5):
        return 0  # N
    elif deg < 67.5:
        return 1  # NE
    elif deg < 112.5:
        return 2  # E
    elif deg < 157.5:
        return 3  # SE
    elif deg < 202.5:
        return 4  # S
    elif deg < 247.5:
        return 5  # SW
    elif deg < 292.5:
        return 6  # W
    else:
        return 7  # NW

In [88]:
vreme['wind_direction'] = vreme['WindDirection'].apply(wind_dir_bin)

In [90]:
vreme.drop(['Rainfall', 'WindDirection'], axis=1)

,season,round,AirTemp,Humidity,Pressure,TrackTemp,WindSpeed,rain_bin,wind_direction
0,2018,1,24.077477,30.915315,997.003604,36.324324,3.691892,1,7
1,2018,2,27.982524,47.363107,1009.494175,32.198058,0.958252,0,4
2,2018,3,19.446429,24.089286,1018.131250,37.019643,1.837500,1,3
3,2018,4,16.661404,45.651754,1021.913158,25.251754,2.222807,0,3
4,2018,5,16.050476,52.286667,1001.541905,32.339048,1.952381,1,2
...,...,...,...,...,...,...,...,...,...
144,2024,20,19.888679,51.050314,785.252830,35.602516,2.150943,0,6
145,2024,21,21.662687,85.258706,926.456716,25.769652,0.735821,1,4
146,2024,22,17.825175,47.860140,935.892308,17.723776,2.898601,0,5
147,2024,23,18.926490,57.251656,1015.500000,22.773510,1.657616,0,2


In [ ]:
trke = trke.merge(vreme, on=['season', 'round'], how='left')    
    

In [91]:
trke['is_sprint_weekend'] = False

In [93]:
#u formatu [season, round]
sprint_vikendi = [
    [2021, 10], 
    [2021, 14], 
    [2021, 19], 

    [2022, 4], 
    [2022, 11], 
    [2022, 21], 

    [2023, 4], 
    [2023, 10], 
    [2023, 13], 
    [2023, 18], 
    [2023, 19], 
    [2023, 21], 

    [2024, 5],
    [2024, 6], 
    [2024, 11], 
    [2024, 19], 
    [2024, 21], 
    [2024, 23] 
]

In [117]:
for i in range(len(trke)):
    if [trke.loc[i]['season'], trke.loc[i]['round']] in sprint_vikendi:
        trke.at[i, 'is_sprint_weekend'] = True